# Diabetes Prediction using Machine Learning

**Goal:** Predict whether a person has diabetes (`diabetes = 1`) based on health-related features.

**Project workflow**
1. Load the dataset
2. Explore and understand the data
3. Clean missing/invalid values and remove duplicates
4. Prepare features for machine learning
5. Train multiple classification models
6. Evaluate models using Accuracy
7. Select the best model
8. Summarize key findings

In [ ]:
# Install dependencies
!pip -q install kagglehub scikit-learn seaborn matplotlib pandas numpy

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
# Load dataset from Kaggle
import kagglehub
from kagglehub import KaggleDatasetAdapter

file_path = ""

df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "iammustafatz/diabetes-prediction-dataset",
    file_path
)

print("Dataset shape:", df.shape)
display(df.head())

In [ ]:
# Basic information
print("Columns:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes)

print("\nDataset information:")
df.info()

print("\nDescriptive statistics:")
display(df.describe(include="all").T)

## 1. Data Exploration
We check:
- Missing values
- Duplicates
- Target distribution
- Numeric distributions
- Relationships between important health variables and diabetes

In [ ]:
# Missing values
missing = df.isnull().sum().sort_values(ascending=False)
print("Missing values:")
display(missing[missing > 0])

# Duplicates
print("Duplicate rows:", df.duplicated().sum())

# Target distribution
print("Target distribution:")
display(df["diabetes"].value_counts())
display(df["diabetes"].value_counts(normalize=True).rename("percentage"))

In [ ]:
# Remove exact duplicate rows
df = df.drop_duplicates().copy()

print("Shape after removing duplicates:", df.shape)

In [ ]:
# Check possible invalid values
numeric_cols = df.select_dtypes(include=np.number).columns

invalid_summary = {}
for col in numeric_cols:
    invalid_summary[col] = {
        "negative_values": int((df[col] < 0).sum()),
        "zero_values": int((df[col] == 0).sum())
    }

invalid_df = pd.DataFrame(invalid_summary).T
display(invalid_df)

In [ ]:
# Target distribution
plt.figure(figsize=(6,4))
sns.countplot(data=df, x="diabetes")
plt.title("Diabetes Class Distribution")
plt.xlabel("Diabetes")
plt.ylabel("Count")
plt.show()

# Numeric distributions
df.hist(figsize=(14,10), bins=25)
plt.suptitle("Numeric Feature Distributions")
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap for numeric features
plt.figure(figsize=(10,7))
corr = df.select_dtypes(include=np.number).corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation Heatmap")
plt.show()

In [ ]:
# Compare important numeric features by diabetes status
important_numeric = [
    col for col in ["age", "bmi", "HbA1c_level", "blood_glucose_level"]
    if col in df.columns
]

for col in important_numeric:
    plt.figure(figsize=(7,4))
    sns.boxplot(data=df, x="diabetes", y=col)
    plt.title(f"{col} vs Diabetes")
    plt.show()

## 2. Data Preparation
The dataset contains both numerical and categorical variables.

For numerical columns:
- Missing values are replaced using the median.
- StandardScaler is used for models that benefit from scaling.

For categorical columns:
- Missing values are replaced using the most frequent value.
- OneHotEncoder converts categories into numerical features.

Using a preprocessing pipeline prevents data leakage because transformations are learned only from the training set.

In [ ]:
# Separate features and target
X = df.drop(columns=["diabetes"])
y = df["diabetes"]

categorical_cols = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
numeric_cols = X.select_dtypes(include=np.number).columns.tolist()

print("Categorical columns:", categorical_cols)
print("Numeric columns:", numeric_cols)

In [ ]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

In [ ]:
# Preprocessing pipelines
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols)
    ]
)

## 3. Build Classification Models
We compare four different classification algorithms:
- Logistic Regression
- Decision Tree
- Random Forest
- K-Nearest Neighbors (KNN)

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    "KNN": KNeighborsClassifier(n_neighbors=5)
}

results = []
trained_models = {}

for name, model in models.items():
    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)

    results.append({
        "Model": name,
        "Accuracy": accuracy
    })

    trained_models[name] = pipeline

results_df = pd.DataFrame(results).sort_values(
    "Accuracy", ascending=False
).reset_index(drop=True)

display(results_df)

In [ ]:
# Accuracy comparison
plt.figure(figsize=(9,5))
sns.barplot(data=results_df, x="Accuracy", y="Model")
plt.xlim(0, 1)
plt.title("Model Accuracy Comparison")
plt.xlabel("Accuracy")
plt.ylabel("Model")
plt.show()

In [ ]:
# Detailed evaluation of the best model
best_model_name = results_df.loc[0, "Model"]
best_model = trained_models[best_model_name]

best_predictions = best_model.predict(X_test)

print("Best model:", best_model_name)
print("Accuracy:", accuracy_score(y_test, best_predictions))

print("\nClassification Report:")
print(classification_report(y_test, best_predictions))

cm = confusion_matrix(y_test, best_predictions)

plt.figure(figsize=(6,5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["No Diabetes", "Diabetes"],
    yticklabels=["No Diabetes", "Diabetes"]
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title(f"Confusion Matrix - {best_model_name}")
plt.show()

## 4. Key Findings
Use the model comparison and visualizations above to summarize the main findings.

Typical observations to discuss:
- `blood_glucose_level` and `HbA1c_level` are expected to be among the most informative health-related features.
- Age, BMI, hypertension, and heart disease can also contribute to diabetes risk.
- The best model is selected based on the highest test Accuracy.
- Accuracy alone should be interpreted carefully when the classes are imbalanced, so the Classification Report and Confusion Matrix are also important.

In [ ]:
# Automatic project summary
print("FINAL PROJECT SUMMARY")
print("=" * 50)
print(f"Final dataset shape: {df.shape}")
print(f"Best model: {best_model_name}")
print(f"Best test accuracy: {results_df.loc[0, 'Accuracy']:.4f}")
print("\nModel ranking:")
display(results_df)

## 5. Optional: Save the Results

The following cell saves the model comparison table as a CSV file that can be used in the presentation.

In [ ]:
results_df.to_csv("diabetes_model_comparison.csv", index=False)
print("Saved: diabetes_model_comparison.csv")